In [1]:
import os  

In [2]:
%pwd

'd:\\ML-Projects\\End-to-end-Machine-Learning-World-Development-Measurement-Clustering-Analysis-with-MLFlow\\notebooks'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\ML-Projects\\End-to-end-Machine-Learning-World-Development-Measurement-Clustering-Analysis-with-MLFlow'

In [5]:
# Components of data validation
import pandas as pd

In [7]:
data = pd.read_excel("artifacts/data_ingestion/cleaned_wdm_data.xlsx")   
data.head()

,birth_rate,business_tax_rate,co2_emissions,country,days_to_start_business,ease_of_business,energy_usage,gdp,health_exp_percent_gdp,health_expcapita,...,life_expectancy_male,mobile_phone_usage,number_of_records,population_014,population_1564,population_65,population_total,population_urban,tourism_inbound,tourism_outbound
0,0.020,NaN,87931.0,Algeria,NaN,NaN,26998.0,"$54,790,058,957",0.035,$60,...,67.0,0.0,1,0.342,0.619,0.039,31719449,0.599,"$102,000,000","$193,000,000"
1,0.050,NaN,9542.0,Angola,NaN,NaN,7499.0,"$9,129,594,819",0.034,$22,...,44.0,0.0,1,0.476,0.499,0.025,13924930,0.324,"$34,000,000","$146,000,000"
2,0.043,NaN,1617.0,Benin,NaN,NaN,1983.0,"$2,359,122,303",0.043,$15,...,53.0,0.0,1,0.454,0.517,0.029,6949366,0.383,"$77,000,000","$50,000,000"
3,0.027,NaN,4276.0,Botswana,NaN,NaN,1836.0,"$5,788,311,645",0.047,$152,...,49.0,0.1,1,0.383,0.587,0.029,1755375,0.532,"$227,000,000","$209,000,000"
4,0.046,NaN,1041.0,Burkina Faso,NaN,NaN,NaN,"$2,610,959,139",0.051,$12,...,49.0,0.0,1,0.468,0.505,0.028,11607944,0.178,"$23,000,000","$30,000,000"


In [8]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2704 entries, 0 to 2703
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   birth_rate              2585 non-null   float64
 1   business_tax_rate       1423 non-null   object 
 2   co2_emissions           2125 non-null   float64
 3   country                 2704 non-null   object 
 4   days_to_start_business  1718 non-null   float64
 5   ease_of_business        185 non-null    float64
 6   energy_usage            1785 non-null   float64
 7   gdp                     2494 non-null   object 
 8   health_exp_percent_gdp  2395 non-null   float64
 9   health_expcapita        2395 non-null   object 
 10  hours_to_do_tax         1416 non-null   float64
 11  infant_mortality_rate   2444 non-null   float64
 12  internet_usage          2531 non-null   float64
 13  lending_interest        1880 non-null   float64
 14  life_expectancy_female  2568 non-null   

In [9]:
data.isnull().sum()

birth_rate                 119
business_tax_rate         1281
co2_emissions              579
country                      0
days_to_start_business     986
ease_of_business          2519
energy_usage               919
gdp                        210
health_exp_percent_gdp     309
health_expcapita           309
hours_to_do_tax           1288
infant_mortality_rate      260
internet_usage             173
lending_interest           824
life_expectancy_female     136
life_expectancy_male       136
mobile_phone_usage         167
number_of_records            0
population_014             220
population_1564            220
population_65              220
population_total             0
population_urban            26
tourism_inbound            368
tourism_outbound           471
dtype: int64

In [10]:
data.shape

(2704, 25)

In [11]:
data.columns

Index(['birth_rate', 'business_tax_rate', 'co2_emissions', 'country',
       'days_to_start_business', 'ease_of_business', 'energy_usage', 'gdp',
       'health_exp_percent_gdp', 'health_expcapita', 'hours_to_do_tax',
       'infant_mortality_rate', 'internet_usage', 'lending_interest',
       'life_expectancy_female', 'life_expectancy_male', 'mobile_phone_usage',
       'number_of_records', 'population_014', 'population_1564',
       'population_65', 'population_total', 'population_urban',
       'tourism_inbound', 'tourism_outbound'],
      dtype='object')

In [12]:
## Numeric and categoriical columns
numeric_features = [feature for feature in data.columns if data[feature].dtype != 'O']
print(f"Numeric Features : {numeric_features}")
print(f"Total Numeric Features : {len(numeric_features)}")

categorical_features = [feature for feature in data.columns if data[feature].dtype == 'O']
print(f"Categorical Features : {categorical_features}")
print(f"Total Categorical Features : {len(categorical_features)}")

Numeric Features : ['birth_rate', 'co2_emissions', 'days_to_start_business', 'ease_of_business', 'energy_usage', 'health_exp_percent_gdp', 'hours_to_do_tax', 'infant_mortality_rate', 'internet_usage', 'lending_interest', 'life_expectancy_female', 'life_expectancy_male', 'mobile_phone_usage', 'number_of_records', 'population_014', 'population_1564', 'population_65', 'population_total', 'population_urban']
Total Numeric Features : 19
Categorical Features : ['business_tax_rate', 'country', 'gdp', 'health_expcapita', 'tourism_inbound', 'tourism_outbound']
Total Categorical Features : 6


In [13]:
## Preparing Entity

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    unzip_data_dir: Path
    STATUS_FILE: str
    VALIDATION_REPORT: Path
    all_schema: dict
    number_of_columns: int


In [14]:
## Configuration

from wdmproject.constants import *
from wdmproject.utils.common import read_yaml, create_directories

In [15]:
## Configuration manager class
class ConfigurationManager:
    def __init__(
        self,
        config_filepath: Path = CONFIG_FILE_PATH,
        params_filepath: Path = PARAMS_FILE_PATH,
        schema_filepath: Path = SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        
        create_directories([self.config.artifacts_root])

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema_columns = self.schema.COLUMNS
        number_of_columns = self.schema.NUMBER_OF_COLUMNS

        create_directories([config.root_dir])
        
        data_validation_config = DataValidationConfig(
            root_dir=Path(config.root_dir),
            unzip_data_dir=Path(config.unzip_data_dir),
            STATUS_FILE=Path(config.STATUS_FILE),
            VALIDATION_REPORT=Path(config.VALIDATION_REPORT),
            all_schema=schema_columns,
            number_of_columns=number_of_columns,
        )
        
        return data_validation_config

In [16]:
import os
from wdmproject import logger
import json
from datetime import datetime
import numpy as np

In [22]:
class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config
        ## Global class variable for dataframe
        self.data = None
        self.validation_report = {}

            
    # Dataset Loading
    def load_data(self):
        try:
            self.data = pd.read_excel(self.config.unzip_data_dir)
            with open(self.config.STATUS_FILE, 'w') as f:
                    f.write(f"Data Loaded Successfully. \n")
            logger.info('Data Loaded Succesfully.')

            self.validation_report['dataset_load'] = {
                 "status" : "Success",
                 "rows" : self.data.shape[0],
                 "columns" : self.data.shape[1]
            }


        except Exception as e:
            logger.error(f"Error Loading Dataset. {e}")
            self.validation_report['dataset_load'] = {
                 "status" : "Failed",
                 "error_msg" : "Error Loading Dataset.",
                 "error" : str(e),
            }
            raise e

    
    ## Some important features in dataset like 'Business Tax Rate', 'GDP', 'Health Exp/Capita', 'Tourism Inbound', 'Tourism Outbound', are
    # considered to be object due to presence of currency symbol in the data except Country 
    # So we will temporarily clean the data for further data validations performed on these features as they are very important for analysis.

    def clean_currency_columns(self):
        try :

            data = self.data

            currency_columns = [
                'business_tax_rate',
                'gdp', 
                'health_expcapita', 
                'tourism_inbound', 
                'tourism_outbound']

            for col in currency_columns:

                if col in data.columns:

                    data[col] = (
                        data[col]
                        .astype(str)
                        .str.replace(r"[^\d.]", "", regex=True)
                        .replace("", np.nan)
                        .astype(float)
                    )
            
            self.data = data

            logger.info("Currency Symbols temporarily cleaned for valiation checks.")

            self.validation_report["currency_cleaning"] = {
                "status" : "Success",
                "columns_cleaned" : currency_columns
            }

        except Exception as e:
            
            logger.error(f"Currency cleaning error : {e}")

            self.validation_report["currency_cleaning"] = {
                "status" : "Failed",
                "error" : str(e)
            }
            
            raise e

    ## Checking wheather all the columns in dataset matches with schema
    def validate_all_columns(self) -> bool:
        try:

            data = self.data
            
            validation_status = None
            
            ## First checking the columns in dataset with expected columns in schema
            if len(data.columns) != self.config.number_of_columns:
                    validation_status = False
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"Data Validation Failed: Expected {self.config.number_of_columns} columns, but found {len(data.columns)} columns.\n")
                    logger. error(f"Expected {self.config.number_of_columns} columns, but found {len(data.columns)} columns.")
                
                    self.validation_report["columns_count"] = {
                        "status" : "Warning",
                        "expected" : self.config.number_of_columns,
                        "actual" : len(data.columns)
                    }
            
            else:
                
                validation_status = True
                self.validation_report["columns_count"] = {
                    "status" : "Success",
                    "expected" : self.config.number_of_columns,
                    "actual" : len(data.columns)
                }
            
                logger.info(f"Column Count Validation Passed. Expected {self.config.number_of_columns} columns, Actual {len(data.columns)} columns.")
                #return validation_status

                ## IF columns lenght matches move forward to check the cloumns with the dtypes

            expected_columns = set(self.config.all_schema.keys())
            actual_columns = set(data.columns)
        
            missing_columns = expected_columns - actual_columns
            extra_columns = actual_columns - expected_columns

               
            ## Checking missing columns        
            if missing_columns:

                validation_status = False
                with open(self.config.STATUS_FILE, 'w') as f:
                    f.write(f"Missing Columns Check Failed: Missing columns: {missing_columns}\n")
                logger.error(f"Missing columns: {missing_columns}")

                self.validation_report["missing_columns"] = {
                    "status" : "Warning",
                    "message" : "Missing Columns Check Failed",
                    "data" : list(missing_columns) 
                }
                

            else:
                validation_status = True
                with open(self.config.STATUS_FILE, 'w') as f:
                    f.write("Missing Columns Check Passed: All expected columns are present.\n")
                
                logger.info("Missing Columns Check Passed: All expected columns are present.\n")

                self.validation_report["missing_columns"] = {
                    "status" : "Success",
                    "message" : "Missing Columns Check Passed"
                }
                

            if extra_columns:
                logger.warning(f"Extra Columns Found: {extra_columns}")

                self.validation_report["extra_columns"] = {
                    "status" : "Warning",
                    "message" : "Extra Columns Found",
                    "data" : list(extra_columns) 
                }
            else:
                logger.info(f"No Extra columns: {extra_columns}")

                self.validation_report["extra_columns"] = {
                    "status" : "Success",
                    "message" : "No Extra Columns Found",
                    "data" : list(extra_columns) 
                }

            ## Validating Datatypes
            dtype_mismatch = []

            for column, expected_dtype in self.config.all_schema.items():

                if column in data.columns:

                    actual_dtype = str(data[column].dtype)

                    if actual_dtype != expected_dtype:

                        dtype_mismatch.append(
                            f"{column}: expected {expected_dtype}, got {actual_dtype}"
                        )
            
            if dtype_mismatch:
                validation_status = False
                logger.error(f"Datatype Mismatch: {dtype_mismatch}")

                self.validation_report["datatype_mismatch"] = {
                    "status" : "Warning",
                    "message" : "Datatype Mismatch in Schema Check Failed",
                    "data" : dtype_mismatch 
                }
            else:
                validation_status = True
                logger.info("No Datatype Mismatch Found in Schema.")

                self.validation_report["datatype_mismatch"] = {
                    "status" : "Success",
                    "message" : "No Datatype Mismatch Found in Schema"
                }

            ## Validation
            with open(self.config.STATUS_FILE, 'w') as f:

                if validation_status:
                    f.write('Validation Passed Successfully. \n')
                else:
                    f.write('Validation Failed Unfortunately. \n')
            
                if missing_columns:
                    f.write(f"Missing Columns : {missing_columns}\n")
                else:
                    f.write(f"No missing columns found!\n")

                if dtype_mismatch:
                    f.write(f"Datatype Mismatch: {dtype_mismatch}\n")
                else:
                    f.write(f"All the datatypes matched with the schema.\n")

        
            return validation_status 
        
        except Exception as e:
            logger.error(f"Data Validation Error: {e}")
            
            self.validation_report["columns_validation"] = {
                    "status" : "Failed",
                    "message" : "Data Validation Error",
                    "error" : str(e)
                }
            
            raise e

        
    
    ## Checking Missing Values
        
    def check_missing_values(self) -> bool:
        try:    
            
            data = self.data
            
            missing = data.isnull().sum()
            
            missing_cols = missing[missing > 0]

            missing_values = missing_cols.to_dict()

            if len(missing_cols) > 0 :
                logger.warning(f"Missing Values Found in : {missing_values}")

                self.validation_report["missing_values_check"] = {
                    "status" : "Warning",
                    "message" : "Missing Values Found.",
                    "columns" : missing_values
                    }

            else:
                logger.info("No Missing Values Found.")

                self.validation_report["missing_values_check"] = {
                    "status" : "Success",
                    "message" : "No Missing Values Found.",
                }

        except Exception as e:
            raise e


    ## Checking Duplicate Rows

    def check_duplicates(self):
        try:
            data = self.data

            duplicates = data.duplicated().sum()

            if duplicates > 0:
                logger.warning(f"Duplicate Rows Found in the Dataset : {duplicates}")
                self.validation_report["duplicates_check"] = {
                    "status" : "Warning",
                    "message" : "Duplicate Rows Found.",
                    "data" : duplicates
                }

            else: 
                logger.info("No Duplicate Rows Found in the Dataset")
                
                self.validation_report["duplicates_check"] = {
                    "status" : "Success",
                    "message" : "No Duplicate Rows Found.",
                }

        except Exception as e:
            raise e


    ## Checking Constant Count

    def check_constant_columns(self):
        try: 
            data = self.data

            constant_cols = [col for col in data.columns if data[col].nunique() <= 1]
            
            if constant_cols:
                logger.warning(f"Constant Columns Found : {constant_cols}")
                self.validation_report["constant_col_check"] = {
                    "status" : "Warning",
                    "message" : "Constant Columns Found.",
                    "columns" : constant_cols
                }
            else:
                logger.info("No Constant Columns Found.")
                self.validation_report["constant_col_check"] = {
                    "status" : "Success",
                    "message" : "No Constant Columns Found."
                }

        except Exception as e:
            raise e
        
    
    ## Validating Features distribution

    def validate_feature_distribution(self):
        try:
            data = self.data

            numeric_cols = data.select_dtypes(include=['float64', 'int64']).columns

            features = {}
            zero_variance_cols = []

            for col in numeric_cols:

                mean = data[col].mean()
                std = data[col].std()
                min = data[col].min()
                max = data[col].max()
                skewness = data[col].skew()
                kurtosis = data[col].kurtosis()

                ## Calculating Z-Score for outliers 
                if std != 0:
                    z_scores = np.abs( (data[col] - mean) / std )
                    outliers = ( z_scores > 3 ).sum()
                    outliers_percentage = ( outliers / len(data[col])) * 100
                else: 
                    outliers_percentage = 0
                    zero_variance_cols.append(col)
                    logger.warning(f"{col} has zero variance.")

                #logger.info(f"{col} -> mean : {mean}, std : {std}")

                features[col] = {
                    "mean" : float(mean),
                    "std" : float(std),
                    "min" : float(min),
                    "max" : float(max),
                    "skewness" : float(skewness),
                    "kurtosis" : float(kurtosis),
                    "outlier_percentage" : round(float(outliers_percentage), 2)
                }


            if zero_variance_cols:
                self.validation_report["feature_distribution"] = {
                    "status" : "Warning",
                    "message" : "Some features have zero variance",
                    "zero_variance_features" : zero_variance_cols,
                    "features" : features,
                }
            else:
                self.validation_report["feature_distribution"] = {
                    "status" : "Success",
                    "message" : "Feature Distribution Validated",
                    "features" : features,
                }


            logger.info("Feature Disctribution validation completed.")

        except Exception as e:
            logger.info("Feature Disctribution validation error: {e}")
            self.validation_report["feature_distribution"] = {
                "status": "Failed",
                "error": str(e)
            }
            raise e


    ## Validating Feature Correlation

    def validate_feature_correlation(self):
        try:
            
            data = self.data

            numeric_cols = data.select_dtypes(include=['float64', 'int64'])

            corr_matrix = numeric_cols.corr()

            threshold = 0.9

            high_corr = []

            for i in range(len(corr_matrix.columns)):
                for j in range(i):

                    corr_value = corr_matrix.iloc[i, j]

                    if abs(corr_value) > threshold:

                        col1 = corr_matrix.columns[i]
                        col2 = corr_matrix.columns[j]

                        high_corr.append({
                            "feature_1" : col1,
                            "feature_2" : col2,
                            "correlation" : round(float(corr_value), 4) 
                        })

            
            if high_corr:
                self.validation_report["feature_correlation"] = {
                    "status" : "Warning",
                    "high_correlated_features" : high_corr
                }

                logger.info(f"Highly Correlated features detected : {high_corr}")
            else:
                self.validation_report["feature_correlation"] = {
                    "status" : "Success",
                    "message" : "No highly correlated features detected."
                }
                logger.info("No highly correlated features detected.")

            logger.info("Feature Correlation Validation Check Completed.")

        except Exception as e:
            logger.error(f"Correlation validation error: {e}")
            raise e


    # Saving Validation Report 
    def save_validation_report(self):

        report_path = self.config.VALIDATION_REPORT

        with open(report_path, "w") as f:

            json.dump(self.validation_report, f, indent=4)

        logger.info(f"Validation report saved at {report_path}")


    ## Initialising Data Validation Pipeline 

    def initiate_data_validation(self):
        try: 
            
            ## Intiating Data Validation stage

            self.validation_report["stage_metadata"] = {
                "stage_name" : "Data Validation",
                "stage_status" : "Running",
                "start_time" : str(datetime.now())
            }

            logger.info("Initiating Data Validation Stage.")

            ## Saving start_time
            start_time = datetime.now()

            ## Loading Dataset
            self.load_data()

            ## Temporary clean currency from columns for validation checks
            self.clean_currency_columns()

            ## Validating all columns, datatypes, and dataframe
            self.validate_all_columns()

            ## Check Missing Values 
            self.check_missing_values()

            ## Checking Duplicates 
            self.check_duplicates()

            ## Checking Constant Columns
            self.check_constant_columns()

            ## Univariate Analysis / Validating Feature Distribution
            self.validate_feature_distribution()

            ## Validating Feature Correlation
            self.validate_feature_correlation()

            ## Stage Success
            self.validation_report["stage_metadata"]["stage_status"] = "Success"
            self.validation_report["stage_metadata"]["end_time"] = str(datetime.now())

            ## Calculating Stage Duration
             
            stage_duration = (datetime.now() - start_time).total_seconds()
            self.validation_report["stage_metadata"]["stage_duration"] = stage_duration
            
            # Saving Validations Report JSON
            self.save_validation_report()

            logger.info("All Data Validation Checks Completed Successfully.")

        except Exception as e:
            
            logger.error(f"Validation Error : {e}")

            end_time = datetime.now()

            self.validation_report["stage_metadata"]["stage_status"] = "Failed"
            self.validation_report["stage_metadata"]["end_time"] = str(end_time)
            self.validation_report["stage_metadata"]["error"] = str(e)

            self.save_validation_report()

            raise e


    

In [23]:
## Data Validation Pipeline
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    data_validation.initiate_data_validation()
    logger.info(f"Data Validation Pipeline Completed Successfully.")
except Exception as e:
    logger.error(f"Data Validation Pipeline Failed: {e}")
    raise e

[2026-03-11 19:38:39,189: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-11 19:38:39,193: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-11 19:38:39,204: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-03-11 19:38:39,208: INFO: common: created directory at: artifacts]
[2026-03-11 19:38:39,212: INFO: common: created directory at: artifacts/data_validation]
[2026-03-11 19:38:39,216: INFO: 1022408613: Initiating Data Validation Stage.]
[2026-03-11 19:38:41,941: INFO: 1022408613: Data Loaded Succesfully.]
[2026-03-11 19:38:42,000: INFO: 1022408613: Currency Symbols temporarily cleaned for valiation checks.]
[2026-03-11 19:38:42,002: INFO: 1022408613: Column Count Validation Passed. Expected 25 columns, Actual 25 columns.]
[2026-03-11 19:38:42,004: INFO: 1022408613: Missing Columns Check Passed: All expected columns are present.
]
[2026-03-11 19:38:42,007: INFO: 1022408613: No Extra columns: set()]
[2026-03-11 19:38:42,011: ERRO